In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
#https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

--2025-03-05 13:51:02--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
2600:9000:2759:4200:b:20a5:b140:21, 2600:9000:2759:9c00:b:20a5:b140:21, 2600:9000:2759:d600:b:20a5:b140:21, ...
verbunden.saufbau zu d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2759:4200:b:20a5:b140:21|:443 … 
HTTP-Anforderung gesendet, auf Antwort wird gewartet … 200 OK
Länge: 64346071 (61M) [binary/octet-stream]
Wird in »yellow_tripdata_2024-10.parquet« gespeichert.

yellow_tripdata_202 100%[===================>]  61,36M  3,25MB/s    in 22s     

2025-03-05 13:51:24 (2,82 MB/s) - »yellow_tripdata_2024-10.parquet« gespeichert [64346071/64346071]



In [3]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/05 13:52:40 WARN Utils: Your hostname, alex-thinkpad resolves to a loopback address: 127.0.1.1; using 192.168.137.153 instead (on interface wlp3s0)
25/03/05 13:52:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/05 13:52:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_yellow = spark.read.parquet('yellow_tripdata_2024-10.parquet')

In [5]:
df_yellow.coalesce(4).write.parquet('data_homework/2024/10/', mode='overwrite')

In [6]:
df_yellow.registerTempTable('yellow')

/home/alex/Downloads/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [9]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [12]:
#How many taxi trips were there on the 15th of October?
spark.sql("""
SELECT
    date_trunc('day', tpep_pickup_datetime) AS day,
    count(1)
FROM
    yellow
GROUP BY 
    day
""").show()

[Stage 5:=============================>                             (2 + 2) / 4]

+-------------------+--------+
|                day|count(1)|
+-------------------+--------+
|2024-11-14 00:00:00|       1|
|2024-10-07 00:00:00|  101075|
|2024-10-01 00:00:00|  108297|
|2024-10-08 00:00:00|  117632|
|2024-10-03 00:00:00|  107463|
|2024-10-05 00:00:00|  121450|
|2024-10-04 00:00:00|  109113|
|2024-10-10 00:00:00|  139571|
|2024-10-24 00:00:00|  133906|
|2024-10-02 00:00:00|  115349|
|2024-10-09 00:00:00|  127465|
|2024-10-11 00:00:00|  131823|
|2024-10-06 00:00:00|  114608|
|2024-10-13 00:00:00|  118919|
|2024-10-16 00:00:00|  133171|
|2024-10-14 00:00:00|   99437|
|2024-10-15 00:00:00|  125567|
|2024-10-12 00:00:00|  127895|
|2024-10-18 00:00:00|  132304|
|2024-10-19 00:00:00|  136038|
+-------------------+--------+
only showing top 20 rows



In [19]:
#What is the length of the longest trip in the dataset in hours?

spark.sql("""
SELECT
    DATEDIFF(hour, tpep_pickup_datetime, tpep_dropoff_datetime) as duration
FROM
    yellow
ORDER by
    duration desc
""").show()

[Stage 9:=============================>                             (2 + 2) / 4]

+--------+
|duration|
+--------+
|     162|
|     143|
|     136|
|     114|
|      89|
|      89|
|      70|
|      67|
|      66|
|      46|
|      42|
|      38|
|      33|
|      26|
|      25|
|      25|
|      24|
|      23|
|      23|
|      23|
+--------+
only showing top 20 rows



In [20]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-05 14:12:47--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
2600:9000:2759:6600:b:20a5:b140:21, 2600:9000:2759:1600:b:20a5:b140:21, 2600:9000:2759:5800:b:20a5:b140:21, ...
Verbindungsaufbau zu d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2759:6600:b:20a5:b140:21|:443 … verbunden.
HTTP-Anforderung gesendet, auf Antwort wird gewartet … 200 OK
Länge: 12331 (12K) [text/csv]
Wird in »taxi_zone_lookup.csv« gespeichert.

taxi_zone_lookup.cs 100%[===================>]  12,04K  --.-KB/s    in 0,006s  

2025-03-05 14:12:48 (2,06 MB/s) - »taxi_zone_lookup.csv« gespeichert [12331/12331]



In [35]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

df_result = df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID)
df_result.registerTempTable('result')

#Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?
spark.sql("""
SELECT
    zone,
    count(1) as freq
FROM
    result
GROUP BY 
    zone
ORDER BY
    freq asc
""").show()

+--------------------+----+
|                zone|freq|
+--------------------+----+
|Governor's Island...|   1|
|       Arden Heights|   2|
|       Rikers Island|   2|
|         Jamaica Bay|   3|
| Green-Wood Cemetery|   3|
|Charleston/Totten...|   4|
|       Port Richmond|   4|
|   Rossville/Woodrow|   4|
|Eltingville/Annad...|   4|
|       West Brighton|   4|
|        Crotona Park|   6|
|         Great Kills|   6|
|Heartland Village...|   7|
|     Mariners Harbor|   7|
|Saint George/New ...|   9|
|             Oakwood|   9|
|New Dorp/Midland ...|  10|
|       Broad Channel|  10|
|         Westerleigh|  12|
|     Pelham Bay Park|  12|
+--------------------+----+
only showing top 20 rows

